# ASB Comparison Analysis — Paired Bootstrap & Exact McNemar

Compare two defenses (e.g. SafeAgent vs baseline) on aligned ASB data:

- **Per-agent comparison** with Wilson 95% CI
- **Paired Bootstrap 95% CI** for the mean difference
- **Exact McNemar Test** for marginal homogeneity of paired binary outcomes

Attack scenarios (DPI / IPI / MP) use `attack_ok` (ASR);
benign scenarios use `task_ok` (PNA).

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Tuple, List
from scipy.stats import binom

import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# Configuration — change these before running
# ============================================================
DEFENSE_A = "SafeAgent"
DEFENSE_B = "baseline"

DATA_DIR = Path("outputs/main_result/ASB")
SCENARIOS = ["benign", "DPI", "IPI", "MP"]

OUTPUT_BOOTSTRAP = "outputs/comparison_bootstrap.csv"
OUTPUT_MCNEMAR = "outputs/comparison_mcnemar.csv"

B = 20000  # bootstrap resamples
ALPHA = 0.05
# ============================================================

In [ ]:
# ============================================================
# Utility functions
# ============================================================

def wilson_ci(count: int, n: int, z: float = 1.96) -> Tuple[float, float]:
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (0.0, 0.0)
    p = count / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt((p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return (centre - margin, centre + margin)


def paired_bootstrap(
    diffs: np.ndarray,
    n_resamples: int = 20000,
    alpha: float = 0.05,
    seed: int = 42,
) -> Tuple[float, float, float]:
    """
    Bootstrap the mean of paired differences.

    Returns (mean_diff, ci_lower, ci_upper).
    """
    rng = np.random.default_rng(seed)
    n = len(diffs)
    means = np.empty(n_resamples)
    for i in range(n_resamples):
        idx = rng.integers(0, n, size=n)
        means[i] = np.mean(diffs[idx])
    lo, hi = np.percentile(means, [alpha / 2 * 100, (1 - alpha / 2) * 100])
    return float(np.mean(diffs)), float(lo), float(hi)


def exact_mcnemar(
    safe_vals: np.ndarray,
    other_vals: np.ndarray,
) -> dict:
    """
    Exact McNemar test for paired binary data.

    Contingency table:
                    Other=1   Other=0
        SafeAgent=1    a        b
        SafeAgent=0    c        d

    H0: b = c  (marginal homogeneity).
    Under H0: min(b,c) ~ Binomial(b+c, 0.5).
    """
    a = int(np.sum((safe_vals == 1) & (other_vals == 1)))
    b = int(np.sum((safe_vals == 1) & (other_vals == 0)))
    c = int(np.sum((safe_vals == 0) & (other_vals == 1)))
    d = int(np.sum((safe_vals == 0) & (other_vals == 0)))

    nd = b + c  # discordant pairs
    if nd == 0:
        return {'a': a, 'b': b, 'c': c, 'd': d,
                'n_discordant': 0, 'p_value': 1.0, 'significant': False}

    k = min(b, c)
    p_value = 2.0 * binom.cdf(k, nd, 0.5)
    p_value = min(p_value, 1.0)

    return {
        'a': a, 'b': b, 'c': c, 'd': d,
        'n_discordant': nd,
        'odds_ratio': b / c if c > 0 else float('inf'),
        'p_value': p_value,
        'significant': p_value < ALPHA,
    }


def load_aligned_scenario(
    defense_a: str,
    defense_b: str,
    scenario: str,
) -> Tuple[np.ndarray, np.ndarray, int]:
    """Load two CSVs and align on (agent_name, task, attacker_idx)."""
    col = "task_ok" if scenario == "benign" else "attack_ok"

    a_df = pd.read_csv(DATA_DIR / defense_a / f"{scenario}.csv")
    b_df = pd.read_csv(DATA_DIR / defense_b / f"{scenario}.csv")

    a_df = a_df[a_df['status'] == 'ok'].reset_index(drop=True)
    b_df = b_df[b_df['status'] == 'ok'].reset_index(drop=True)

    merged = a_df[[col, 'agent_name', 'task', 'attacker_idx']].merge(
        b_df[[col, 'agent_name', 'task', 'attacker_idx']],
        on=['agent_name', 'task', 'attacker_idx'],
        suffixes=('_a', '_b'),
    )
    return merged[f'{col}_a'].values, merged[f'{col}_b'].values, len(merged)


def format_diff(val: float) -> str:
    return f"{val:+.4f}"


print("Functions loaded OK")

In [ ]:
# ============================================================
# 1. Paired Bootstrap — mean difference with 95% CI
# ============================================================
bootstrap_results = []

for scenario in SCENARIOS:
    vals_a, vals_b, n = load_aligned_scenario(DEFENSE_A, DEFENSE_B, scenario)
    metric = "PNA" if scenario == "benign" else "ASR"

    mean_a, mean_b = float(vals_a.mean()), float(vals_b.mean())
    diffs = vals_a - vals_b
    diff_mean, ci_lo, ci_hi = paired_bootstrap(diffs, n_resamples=B, alpha=ALPHA)

    bootstrap_results.append({
        'scenario': scenario,
        'metric': metric,
        'N': n,
        f'{DEFENSE_A}_{metric}': mean_a,
        f'{DEFENSE_B}_{metric}': mean_b,
        'diff_mean': diff_mean,
        'CI_lower': ci_lo,
        'CI_upper': ci_hi,
        'significant': (ci_lo > 0) or (ci_hi < 0),
        'display': f"{diff_mean:+.4f} [{ci_lo:+.4f}, {ci_hi:+.4f}]",
    })

df_bootstrap = pd.DataFrame(bootstrap_results)
print("Paired bootstrap complete.")

In [ ]:
# ============================================================
# 2. Exact McNemar Test
# ============================================================
mcnemar_results = []

for scenario in SCENARIOS:
    vals_a, vals_b, n = load_aligned_scenario(DEFENSE_A, DEFENSE_B, scenario)
    metric = "PNA" if scenario == "benign" else "ASR"
    m = exact_mcnemar(vals_a, vals_b)

    mcnemar_results.append({
        'scenario': scenario,
        'metric': metric,
        'N': n,
        'a (both=1)': m['a'],
        f'b ({DEFENSE_A}=1,{DEFENSE_B}=0)': m['b'],
        f'c ({DEFENSE_A}=0,{DEFENSE_B}=1)': m['c'],
        'd (both=0)': m['d'],
        'n_discordant': m['n_discordant'],
        'odds_ratio': m['odds_ratio'],
        'p_value': m['p_value'],
        'significant': m['significant'],
    })

df_mcnemar = pd.DataFrame(mcnemar_results)
print("Exact McNemar complete.")

In [ ]:
# ============================================================
# Display: Bootstrap
# ============================================================
print("")
print("=" * 70)
print(f"  Paired Bootstrap 95% CI  ({DEFENSE_A} — {DEFENSE_B})")
print("=" * 70)
display(df_bootstrap[[
    'scenario', 'metric', 'N',
    f'{DEFENSE_A}_'+('PNA' if 'benign' in SCENARIOS else 'ASR'),
    f'{DEFENSE_B}_'+('PNA' if 'benign' in SCENARIOS else 'ASR'),
    'diff_mean', 'CI_lower', 'CI_upper', 'significant'
]] if False else df_bootstrap.style.format({
    'diff_mean': '{:+.4f}',
    'CI_lower': '{:+.4f}',
    'CI_upper': '{:+.4f}',
}).background_gradient(subset=['diff_mean'], cmap='RdBu_r'))

print()
print("=" * 70)
print(f"  Exact McNemar Test  ({DEFENSE_A} vs {DEFENSE_B})")
print("=" * 70)
display(df_mcnemar.style.format({
    'p_value': '{:.6f}',
    'odds_ratio': '{:.4f}',
}))

In [ ]:
# ============================================================
# Visualisation: Bootstrap (upper) + McNemar (lower)
# ============================================================
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})

fig, axes = plt.subplots(2, len(SCENARIOS), figsize=(5 * len(SCENARIOS), 7))

if len(SCENARIOS) == 1:
    axes = axes.reshape(2, 1)

colours = plt.cm.Set2(np.linspace(0, 1, len(df_bootstrap)))

for i, scenario in enumerate(SCENARIOS):
    # ---- Upper: Bootstrap - diff ----
    ax = axes[0, i]
    row = df_bootstrap[df_bootstrap['scenario'] == scenario].iloc[0]

    err_l = row['diff_mean'] - row['CI_lower']
    err_u = row['CI_upper'] - row['diff_mean']

    ax.barh(0, row['diff_mean'], 0.4,
            xerr=[[err_l], [err_u]],
            color=colours[i], capsize=4)
    ax.axvline(0, color='grey', ls='--', lw=0.8)
    ax.set_yticks([])
    ax.set_title(f'{scenario}', fontsize=13, fontweight='bold')
    ax.set_xlabel(f'{DEFENSE_A} — {DEFENSE_B}')

    if row['significant']:
        ax.text(0.98, 0.95, '*', transform=ax.transAxes,
                fontsize=18, fontweight='bold', color='green', va='top', ha='right')

    # ---- Lower: McNemar -log10(p) ----
    ax = axes[1, i]
    row_m = df_mcnemar[df_mcnemar['scenario'] == scenario].iloc[0]
    neg_log_p = -np.log10(max(row_m['p_value'], 1e-300))
    sig_col = '#e74c3c' if row_m['significant'] else '#95a5a6'

    ax.barh(0, neg_log_p, 0.4, color=sig_col)
    ax.axvline(-np.log10(ALPHA), color='green', ls='--', lw=1, label=f'p={ALPHA}')
    ax.set_yticks([])
    ax.set_xlabel('-log₁₀(p-value)')
    ax.legend(fontsize=8, loc='lower right')

plt.suptitle(f'{DEFENSE_A} vs {DEFENSE_B}',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Export results
# ============================================================
df_bootstrap.to_csv(OUTPUT_BOOTSTRAP, index=False)
df_mcnemar.to_csv(OUTPUT_MCNEMAR, index=False)
print(f"Bootstrap: {OUTPUT_BOOTSTRAP}")
print(f"McNemar:   {OUTPUT_MCNEMAR}")